In [8]:
import sys
from dynamixel_sdk import *  # 由 Dynamixel SDK 提供

In [9]:
#------------------------------------------------------------------------------
#    偵測目前連線馬達之ID號碼
#    注意:需要只連接單顆馬達情況下
#------------------------------------------------------------------------------
DEVICENAME = "COM4"         # 串口裝置，如 Windows: 'COM3', Linux: '/dev/ttyUSB0'
BAUDRATE =115200      # 與馬達設定一致的波特率 (常見 57600, 115200, 1000000, etc.)
PROTOCOL_VERSION = 2.0      # 多數 Dynamixel Pro / X 系列預設支援 Protocol 2.0

def detectormotorid():
    # 初始化埠控制 (PortHandler)
    portHandler = PortHandler(DEVICENAME)
    # 初始化封包處理 (PacketHandler)
    packetHandler = PacketHandler(PROTOCOL_VERSION)

    # 開啟串口
    if not portHandler.openPort():
        print(f"無法開啟串口: {DEVICENAME}")
        sys.exit(1)

    # 設定波特率
    if not portHandler.setBaudRate(BAUDRATE):
        print(f"無法設定波特率: {BAUDRATE}")
        sys.exit(1)

    print(f"開始掃描 Dynamixel (Protocol {PROTOCOL_VERSION}), BaudRate: {BAUDRATE}")
    found_any = False

    # 嘗試 Ping 所有可能的 ID (0~253)
    for dxl_id in range(0, 254):
        dxl_model_number, dxl_comm_result, dxl_error = packetHandler.ping(portHandler, dxl_id)
        if dxl_comm_result == COMM_SUCCESS:  # 表示成功 Ping 到
            found_any = True
            print(f"【偵測到馬達】ID: {dxl_id}, Model Number: {dxl_model_number}")
        if dxl_error != 0:
            print(f"馬達回傳錯誤: {packetHandler.getRxPacketError(dxl_error)}")

    if not found_any:
        print("尚未偵測到任何馬達，可能是:")
        print("1) 馬達未上電或電源不足")
        print("2) 馬達 Baud Rate 與此程式設定不一致")
        print("3) 馬達 Protocol 與程式指定不同")
        print("4) 接線問題 (RS-485 A/B 未正確, GND 未共地 等)")

    portHandler.closePort()

detectormotorid()

開始掃描 Dynamixel (Protocol 2.0), BaudRate: 115200
【偵測到馬達】ID: 11, Model Number: 51200
馬達回傳錯誤: [RxPacketError] Hardware error occurred. Check the error at Control Table (Hardware Error Status)!
【偵測到馬達】ID: 12, Model Number: 51200
馬達回傳錯誤: [RxPacketError] Hardware error occurred. Check the error at Control Table (Hardware Error Status)!
【偵測到馬達】ID: 13, Model Number: 51200
馬達回傳錯誤: [RxPacketError] Hardware error occurred. Check the error at Control Table (Hardware Error Status)!
【偵測到馬達】ID: 14, Model Number: 51200
馬達回傳錯誤: [RxPacketError] Hardware error occurred. Check the error at Control Table (Hardware Error Status)!


In [150]:
#------------------------------------------------------------------------------
#    改變馬達ID
#    注意:要改變需要之前馬達的ID號碼，不知道可以用上一支程式，同樣只連接單顆馬達情況下
#------------------------------------------------------------------------------
DEVICENAME = 'COM4'         # 串口裝置，如 Windows: 'COM3', Linux: '/dev/ttyUSB0'
BAUDRATE = 115200           # 與馬達設定一致的波特率 (常見 57600, 115200, 1000000, etc.)
PROTOCOL_VERSION = 2.0      # 多數 Dynamixel Pro / X 系列預設支援 Protocol 2.0

ADDR_PRO_ID = 7

# 你想要改變的舊 ID 與新 ID
OLD_ID = 15
NEW_ID = 13

def changedxlid():
    portHandler = PortHandler(DEVICENAME)
    packetHandler = PacketHandler(PROTOCOL_VERSION)


    if portHandler.openPort():
        print(f"成功開啟串口: {DEVICENAME}")
    else:
        print(f"無法開啟串口: {DEVICENAME}")
        sys.exit(1)


    if portHandler.setBaudRate(BAUDRATE):
        print(f"成功設定波特率: {BAUDRATE}")
    else:
        print(f"無法設定波特率: {BAUDRATE}")
        sys.exit(1)

    #------------------------------------------------------------------------------
    # 寫入控制表：將舊ID -> 新ID
    # write1ByteTxRx(...) 用來寫入 1 byte
    #   參數分別是: (埠控制, 舊ID, 地址, 要寫入的值)
    #------------------------------------------------------------------------------
    dxl_comm_result, dxl_error = packetHandler.write1ByteTxRx(portHandler, OLD_ID, ADDR_PRO_ID, NEW_ID)

    # 檢查回應
    if dxl_comm_result != COMM_SUCCESS:
        print(f"通訊失敗: {packetHandler.getTxRxResult(dxl_comm_result)}")
    elif dxl_error != 0:
        print(f"馬達回傳錯誤: {packetHandler.getRxPacketError(dxl_error)}")
    else:
        print(f"成功將 ID 從 {OLD_ID} 改為 {NEW_ID}！")

    portHandler.closePort()

changedxlid()

成功開啟串口: COM4
成功設定波特率: 115200
馬達回傳錯誤: [RxPacketError] Hardware error occurred. Check the error at Control Table (Hardware Error Status)!


In [80]:
DEVICENAME       = 'COM4'
PROTOCOL_VERSION = 2.0
DXL_ID           = 6                # 目標馬達 ID
ADDR_BAUDRATE    = 8

# 舊 Baud Rate (程式用來連線)
OLD_BAUDRATE     = 1000000
# 新 Baud Rate => 115200 bps，對應 Index = 2
NEW_BAUDRATE_INDEX = 2
# 方便後續測試，對照表得知 Index=2 對應 115200
TEST_BAUDRATE    = 115200

def main():
    portHandler = PortHandler(DEVICENAME)
    packetHandler = PacketHandler(PROTOCOL_VERSION)

    if not portHandler.openPort():
        print(f"無法開啟串口: {DEVICENAME}")
        sys.exit(1)
    if not portHandler.setBaudRate(OLD_BAUDRATE):
        print(f"無法設定舊 Baud Rate: {OLD_BAUDRATE}")
        sys.exit(1)

    print(f"已開啟串口: {DEVICENAME}, BaudRate={OLD_BAUDRATE}")
    print("嘗試 Ping 目標馬達...")
    dxl_model_num, dxl_comm_result, dxl_error = packetHandler.ping(portHandler, DXL_ID)
    if dxl_comm_result == COMM_SUCCESS:
        print(f"成功 Ping 到馬達 ID:{DXL_ID}, 型號: {dxl_model_num}")
    else:
        print(f"Ping 失敗: {packetHandler.getTxRxResult(dxl_comm_result)}")
        print("請檢查接線、ID、舊 BaudRate 是否正確。")
        portHandler.closePort()
        sys.exit(1)

    print(f"開始寫入新的 Baud Rate 索引 (Index={NEW_BAUDRATE_INDEX}) 到地址 {ADDR_BAUDRATE} ...")
    dxl_comm_result, dxl_error = packetHandler.write1ByteTxRx(portHandler, DXL_ID, ADDR_BAUDRATE, NEW_BAUDRATE_INDEX)
    if dxl_comm_result != COMM_SUCCESS:
        print(f"寫入失敗: {packetHandler.getTxRxResult(dxl_comm_result)}")
    elif dxl_error != 0:
        print(f"硬體錯誤: {packetHandler.getRxPacketError(dxl_error)}")
    else:
        print("成功寫入新 Baud Rate Index!")
    # 馬達現在內部已切換到新的 Baud Rate；若此後再用舊 Baud Rate 連線就會失敗。

    portHandler.closePort()
    print("已關閉通訊埠。")

    #-----------------------------
    # 使用新的 Baud Rate 測試
    #-----------------------------
    print(f"\n改用新的 Baud Rate ({TEST_BAUDRATE}) 重新嘗試 Ping ID:{DXL_ID} ...")
    portHandler2 = PortHandler(DEVICENAME)
    if not portHandler2.openPort():
        print(f"無法開啟串口: {DEVICENAME}")
        sys.exit(1)
    if not portHandler2.setBaudRate(TEST_BAUDRATE):
        print(f"無法設定新 Baud Rate: {TEST_BAUDRATE}")
        sys.exit(1)

    packetHandler2 = PacketHandler(PROTOCOL_VERSION)

    dxl_model_num, dxl_comm_result, dxl_error = packetHandler2.ping(portHandler2, DXL_ID)
    if dxl_comm_result == COMM_SUCCESS:
        print(f"成功以新 Baud Rate ({TEST_BAUDRATE}) Ping 到馬達 ID:{DXL_ID}, 型號: {dxl_model_num}")
    else:
        print(f"Ping 失敗，可能新 Baud Rate 與馬達實際設定不符 (或 ID 不符)。")
        print(f"原始錯誤: {packetHandler2.getTxRxResult(dxl_comm_result)}")

    portHandler2.closePort()

main()


已開啟串口: COM4, BaudRate=1000000
嘗試 Ping 目標馬達...
成功 Ping 到馬達 ID:6, 型號: 54024
開始寫入新的 Baud Rate 索引 (Index=2) 到地址 8 ...
硬體錯誤: [RxPacketError] Hardware error occurred. Check the error at Control Table (Hardware Error Status)!
已關閉通訊埠。

改用新的 Baud Rate (115200) 重新嘗試 Ping ID:6 ...
成功以新 Baud Rate (115200) Ping 到馬達 ID:6, 型號: 54024


In [155]:
DEVICENAME           = 'COM4'   # 串口裝置 (Windows: 'COM3', Linux: '/dev/ttyUSB0')
BAUDRATE             = 115200          # 與馬達設定一致 (常見為 57600, 115200, 1000000 ...)
PROTOCOL_VERSION     = 2.0
DXL_ID               = 14              # 目標馬達 ID
ADDR_HARDWARE_ERROR  = 70              # Hardware Error Status 在控制表的位址

def main():
    portHandler = PortHandler(DEVICENAME)
    packetHandler = PacketHandler(PROTOCOL_VERSION)

    if not portHandler.openPort():
        print(f"無法開啟串口: {DEVICENAME}")
        sys.exit(1)

    if not portHandler.setBaudRate(BAUDRATE):
        print(f"無法設定波特率: {BAUDRATE}")
        sys.exit(1)

    print(f"嘗試讀取馬達(ID={DXL_ID})的 Hardware Error Status...")

    dxl_hardware_error, dxl_comm_result, dxl_error = packetHandler.read1ByteTxRx(portHandler, DXL_ID, ADDR_HARDWARE_ERROR)

    if dxl_comm_result != COMM_SUCCESS:
        print(f"通訊失敗: {packetHandler.getTxRxResult(dxl_comm_result)}")
    elif dxl_error != 0:
        print(f"馬達回傳錯誤: {packetHandler.getRxPacketError(dxl_error)}")
    if dxl_error == 0:
        print("Hardware Error Status = 0 (沒有偵測到任何硬體錯誤)")
    else:
        print(f"Hardware Error Status = 0x{dxl_hardware_error:02X} (二進位: {dxl_hardware_error:08b})")
        if dxl_hardware_error & 0x01:
            print(" - [Bit 0] Input Voltage Error (電壓錯誤)")
        if dxl_hardware_error & 0x02:
            print(" - [Bit 1] Overheating Error (溫度過高)")
        if dxl_hardware_error & 0x04:
            print(" - [Bit 2] Motor Encoder Error (編碼器錯誤)")
        if dxl_hardware_error & 0x08:
            print(" - [Bit 3] Electrical Shock Error (電子衝擊/短路)")
        if dxl_hardware_error & 0x10:
            print(" - [Bit 4] Overload Error (過載)")
    '''
    ADDR_PRESENT_TEMP =146
    present_temp, comm_result, error = packetHandler.read1ByteTxRx(portHandler, DXL_ID, ADDR_PRESENT_TEMP)
    if comm_result == COMM_SUCCESS and error == 0:
        print(f"讀到馬達的溫度: {present_temp} °C")
    '''
    portHandler.closePort()


main()


嘗試讀取馬達(ID=14)的 Hardware Error Status...
馬達回傳錯誤: [RxPacketError] Hardware error occurred. Check the error at Control Table (Hardware Error Status)!
Hardware Error Status = 0x02 (二進位: 00000010)
 - [Bit 1] Overheating Error (溫度過高)


In [ ]:
import cv2
print(cv2.__file__)
print(hasattr(cv2, "data"))  # True
print(cv2.data.haarcascades)

: 